In [1]:
!pip install datasets transformers torch accelerate

   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 5.9 MB/s  0:00:00

   ------ ---------------------------------  2/12 [multidict]
  Attempting uninstall: fsspec
   ------ ---------------------------------  2/12 [multidict]
    Found existing installation: fsspec 2026.7.0
   ------ ---------------------------------  2/12 [multidict]
   ---------- -----------------------------  3/12 [fsspec]
    Uninstalling fsspec-2026.7.0:
   ---------- -----------------------------  3/12 [fsspec]
      Successfully uninstalled fsspec-2026.7.0
   ---------- -----------------------------  3/12 [fsspec]
   ---------- -----------------------------  3/12 [fsspec]
   ---------- -----------------------------  3/12 [fsspec]
   ---------- -----------------------------  3/12 [fsspec]
   ---------- -----------------------------  3/12 [fsspec]
   ---------- -----------------------------  3/12 [fsspec]
   ---------- ------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import torch

# ==========================================
# EXPERIMENT 10: FINE-TUNING
# ==========================================

# Small training dataset
data = {
    "text": [
        "I love this product",
        "This movie is amazing",
        "The service was excellent",
        "I am very happy with this",
        "I hate this product",
        "This movie is terrible",
        "The service was very bad",
        "I am disappointed with this"
    ],
    
    # 1 = Positive, 0 = Negative
    "label": [1, 1, 1, 1, 0, 0, 0, 0]
}

# Create dataset
dataset = Dataset.from_dict(data)

# Load lightweight DistilBERT model
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

# Training configuration
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("Starting fine-tuning...\n")

# Fine-tune the model
trainer.train()

print("\nFine-tuning completed successfully!")

# ==========================================
# TEST THE FINE-TUNED MODEL
# ==========================================

test_text = "I really enjoyed this product"

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Predict
with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

if prediction == 1:
    result = "POSITIVE"
else:
    result = "NEGATIVE"

print("\nTest Sentence:", test_text)
print("Predicted Sentiment:", result)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            